# ASG Airlines — End-to-End Data Engineering Pipeline

**Case Study:** ASG Airlines Flight Operations Data Pipeline
**Author:** Jii
**Notebook purpose:** Ingest → Validate → Clean → Transform → Model → KPI-ready curated data for Power BI

---

## What this notebook does

1. **Ingestion** — loads all 4 source tables (`flights`, `bookings`, `passengers`, `payments`) from the raw Excel workbook.
2. **Profiling** — establishes ground-truth data quality metrics (nothing is assumed, everything is measured).
3. **Data Contract & Validation** — defines what a "valid" record means per table, tags every row with a validation status and reason code.
4. **Cleaning & Standardization** — fixes casing/whitespace issues, standardizes categorical values, handles missing values with documented rules.
5. **Overnight Flight Handling** — correctly computes flight duration across midnight boundaries and flags genuinely invalid time records (e.g. arrival earlier than departure by more than a day is possible).
6. **PII Masking** — hashes/masks passenger-sensitive fields (Aadhaar ID, passport number, phone, email, emergency contact) before they reach the analytics layer.
7. **Dimensional Model** — builds a star schema (`FactFlights` + `DimAirline`, `DimRoute`, `DimDate`, plus supporting `DimPassenger`, `FactBookings`, `FactPayments`).
8. **KPIs** — Average Flight Duration, Route-wise Traffic, Airline Distribution, Delay/Anomaly rate, Data Quality Score, Revenue KPIs.
9. **Quarantine Layer** — every rejected record is preserved with a reason, never silently dropped.
10. **Testing** — unit tests for the cleaning/transformation logic (overnight flights, missing values, malformed IDs, duplicates, etc.).
11. **Pipeline Run Log** — a structured summary block, the way a production pipeline would report a run.


## 0. Setup & Configuration

In [1]:
import pandas as pd
import numpy as np
import re
import hashlib
import warnings
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

# ---- Project paths ----
BASE_DIR      = Path('..')
RAW_DIR       = BASE_DIR / 'data' / 'raw'
CLEANED_DIR   = BASE_DIR / 'data' / 'cleaned'
CURATED_DIR   = BASE_DIR / 'data' / 'curated'
QUARANTINE_DIR = BASE_DIR / 'data' / 'quarantine'

for d in [RAW_DIR, CLEANED_DIR, CURATED_DIR, QUARANTINE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_FILE = BASE_DIR / 'data' / 'sample' / 'UseCase_-_Airlines_sample.xlsx'
# For the competition source, replace the path above with data/raw/UseCase_-_Airlines.xlsx locally.
# Never commit the raw competition workbook to GitHub.

# A simple run-time log we will append to throughout the notebook,
# then print as a single structured pipeline report at the end (Section 12).
PIPELINE_LOG = {}

print('Setup complete. Raw file found:', RAW_FILE.exists())


Setup complete. Raw file found: True


## 1. Data Ingestion

ASG Airlines' data spans four operational systems, all provided as sheets in a single workbook:

| Sheet | Represents | Source system (conceptually) |
|---|---|---|
| `flights` | Flight schedule records | Scheduling system / airport logs |
| `bookings` | Passenger bookings per flight | Booking platform |
| `passengers` | Passenger master data (contains PII) | Booking platform / CRM |
| `payments` | Payment transactions per booking | Payment gateway |

We ingest all four "as-is" into a **raw layer** first — the raw file is never modified, so the pipeline stays reproducible and auditable.


In [2]:
xls = pd.ExcelFile(RAW_FILE)
print('Sheets found:', xls.sheet_names)

flights_raw    = pd.read_excel(xls, 'flights')
bookings_raw   = pd.read_excel(xls, 'bookings')
passengers_raw = pd.read_excel(xls, 'passengers')
payments_raw   = pd.read_excel(xls, 'payments')

# Normalize datetime columns immediately after ingestion so profiling and transformations are type-safe.
for _df in (flights_raw, bookings_raw, passengers_raw, payments_raw):
    for _col in ('departure_time', 'arrival_time'):
        if _col in _df.columns:
            _df[_col] = pd.to_datetime(_df[_col], errors='coerce')

print()
for name, df in [('flights', flights_raw), ('bookings', bookings_raw),
                  ('passengers', passengers_raw), ('payments', payments_raw)]:
    print(f'{name:12s} -> {df.shape[0]:5d} rows x {df.shape[1]} columns')


Sheets found: ['flights', 'bookings', 'passengers', 'payments']

flights      ->   100 rows x 7 columns
bookings     ->   100 rows x 9 columns
passengers   ->   200 rows x 9 columns
payments     ->    94 rows x 4 columns


In [3]:
# Persist a local landing/raw copy for reproducibility. The public repository excludes data/raw/.
# This is the layer we NEVER overwrite -- every later transform reads from here or from
# an explicit copy, never mutates these files in place.
flights_raw.to_csv(RAW_DIR / 'flights_raw.csv', index=False)
bookings_raw.to_csv(RAW_DIR / 'bookings_raw.csv', index=False)
passengers_raw.to_csv(RAW_DIR / 'passengers_raw.csv', index=False)
payments_raw.to_csv(RAW_DIR / 'payments_raw.csv', index=False)

PIPELINE_LOG['ingestion'] = {
    'flights_rows': len(flights_raw),
    'bookings_rows': len(bookings_raw),
    'passengers_rows': len(passengers_raw),
    'payments_rows': len(payments_raw),
}
print('Raw layer written to data/raw/. Rows ingested:', PIPELINE_LOG['ingestion'])


Raw layer written to data/raw/. Rows ingested: {'flights_rows': 100, 'bookings_rows': 100, 'passengers_rows': 200, 'payments_rows': 94}


## 2. Data Profiling (measured, not assumed)

Before writing a single cleaning rule we measure exactly what is wrong with the data. All numbers below come directly from the dataset.


In [4]:
flights_raw.info()
print()
flights_raw.head(5)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   flight_id       100 non-null    object        
 1   airline         97 non-null     object        
 2   source          100 non-null    object        
 3   destination     100 non-null    object        
 4   departure_time  100 non-null    datetime64[ns]
 5   arrival_time    100 non-null    datetime64[ns]
 6   duration        100 non-null    object        
dtypes: datetime64[ns](2), object(5)
memory usage: 5.6+ KB



,flight_id,airline,source,destination,departure_time,arrival_time,duration
0,SJ010,SpiceJet,CCU,MAA,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,02:54:00
1,AI155,Air India,BOM,CCU,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,01:48:00
2,UK094,Vistara,BOM,CCU,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,01:45:00
3,AI245,Air India,BOM,CCU,2026-04-20 23:07:41.704,2026-04-21 01:43:41.704,02:36:00
4,AI192,Air India,MAA,BOM,2026-04-20 23:05:41.703,2026-04-21 04:04:41.703,04:59:00


In [5]:
print('--- FLIGHTS PROFILE ---')
print('Total records            :', len(flights_raw))
print('Total columns            :', flights_raw.shape[1])
print('Fully duplicate rows     :', flights_raw.duplicated().sum())
print('Duplicate flight_id rows :', flights_raw['flight_id'].duplicated().sum())
print('Missing flight_id        :', flights_raw['flight_id'].isna().sum())
print('Missing/blank airline    :', (flights_raw['airline'].isna() | (flights_raw['airline'].astype(str).str.strip()=='')).sum())
print('Missing source           :', flights_raw['source'].isna().sum())
print('Missing destination      :', flights_raw['destination'].isna().sum())
print('Missing departure_time   :', flights_raw['departure_time'].isna().sum())
print('Missing arrival_time     :', flights_raw['arrival_time'].isna().sum())
print()
print('Distinct airline values (raw)    :', sorted(flights_raw['airline'].dropna().unique().tolist()))
print('Distinct source values           :', sorted(flights_raw['source'].unique().tolist()))
print('Distinct destination values      :', sorted(flights_raw['destination'].unique().tolist()))
print()
raw_gap = (flights_raw['arrival_time'] - flights_raw['departure_time']).dt.total_seconds() / 60
print('Rows where arrival < departure (naive, before overnight logic):', (raw_gap < 0).sum())
print('Rows where arrival date != departure date (candidate overnight):', (flights_raw['arrival_time'].dt.date != flights_raw['departure_time'].dt.date).sum())
print('Min naive gap (minutes):', raw_gap.min(), '   Max naive gap (minutes):', raw_gap.max())


--- FLIGHTS PROFILE ---
Total records            : 100
Total columns            : 7
Fully duplicate rows     : 1
Duplicate flight_id rows : 1
Missing flight_id        : 0
Missing/blank airline    : 3
Missing source           : 0
Missing destination      : 0
Missing departure_time   : 0
Missing arrival_time     : 0

Distinct airline values (raw)    : ['Air India', 'IndiGo', 'SpiceJet', 'UNKNOWN', 'Vistara']
Distinct source values           : ['BLR', 'BOM', 'CCU', 'DEL', 'HYD', 'MAA']
Distinct destination values      : ['BLR', 'BOM', 'CCU', 'DEL', 'HYD', 'MAA']

Rows where arrival < departure (naive, before overnight logic): 0
Rows where arrival date != departure date (candidate overnight): 29
Min naive gap (minutes): 30.0    Max naive gap (minutes): 299.0


**Reading the profile:**

The profile above is the source of truth for the dataset. The pipeline does not hard-code record counts or quality findings into the transformation logic. Candidate overnight rows are identified from the parsed datetimes; genuinely invalid time sequences are handled by the validation rules rather than being silently shifted.

The city-code fields are inspected before standardisation so that genuinely distinct locations are not accidentally merged. The same measured-first approach is applied to bookings, passengers and payments in the following profile section.


In [6]:
print('--- BOOKINGS PROFILE ---')
print('Total records          :', len(bookings_raw))
print('Missing status         :', bookings_raw['status'].isna().sum())
print('Status values          :', bookings_raw['status'].dropna().unique().tolist())
print('Duplicate rows         :', bookings_raw.duplicated().sum())
print('Duplicate booking_id   :', bookings_raw['booking_id'].duplicated().sum())
print('Orphan flight_id (not in flights)     :', (~bookings_raw['flight_id'].isin(flights_raw['flight_id'])).sum())
print('Orphan passenger_id (not in passengers):', (~bookings_raw['passenger_id'].isin(passengers_raw['passenger_id'])).sum())

print()
print('--- PASSENGERS PROFILE ---')
print('Total records          :', len(passengers_raw))
print('Missing last_name      :', passengers_raw['last_name'].isna().sum())
print('Duplicate rows         :', passengers_raw.duplicated().sum())
print('Age range              :', passengers_raw['age'].min(), '-', passengers_raw['age'].max())
print('Aadhaar / passport / phone / email are PII -> must be masked before analytics (see Section 6).')

print()
print('--- PAYMENTS PROFILE ---')
print('Total records          :', len(payments_raw))
print('Missing amount         :', payments_raw['amount'].isna().sum())
print("Literal 'INVALID' amount:", (payments_raw['amount'].astype(str)=='INVALID').sum())
print('Payment methods        :', payments_raw['payment_method'].unique().tolist())
print('Orphan booking_id      :', (~payments_raw['booking_id'].isin(bookings_raw['booking_id'])).sum())


--- BOOKINGS PROFILE ---
Total records          : 100
Missing status         : 3
Status values          : ['CANCELLED', 'CONFIRMED', 'PENDING', 'INVALID']
Duplicate rows         : 0
Duplicate booking_id   : 0
Orphan flight_id (not in flights)     : 0
Orphan passenger_id (not in passengers): 0

--- PASSENGERS PROFILE ---
Total records          : 200
Missing last_name      : 0
Duplicate rows         : 0
Age range              : 1 - 89
Aadhaar / passport / phone / email are PII -> must be masked before analytics (see Section 6).

--- PAYMENTS PROFILE ---
Total records          : 94
Missing amount         : 4
Literal 'INVALID' amount: 4
Payment methods        : ['UPI', 'CARD', 'NETBANKING']
Orphan booking_id      : 0


**Reading the profile:**

- `bookings.status` has 45 nulls and a literal `'INVALID'` category alongside the legitimate `CONFIRMED` / `CANCELLED` / `PENDING` values.
- No orphan foreign keys anywhere (every `flight_id`, `passenger_id`, `booking_id` referenced actually exists) — referential integrity is intact, which we still verify rather than assume.
- `passengers.last_name` has 10 nulls — non-critical for flight-ops analytics, so it is retained with a quality flag rather than quarantined.
- `payments.amount` is stored as a mixed-type column: numeric values, missing values, **and** the literal string `'INVALID'` — three different failure states that need three different handling rules.


## 3. Data Contract

Before cleaning anything, we fix down *what a valid record means* for each table. This is the contract the validation layer (Section 4) enforces mechanically.

### `flights`

| Field | Nullable? | Validation rule | Failure action |
|---|---|---|---|
| `flight_id` | No | Non-empty, matches `[A-Z0-9]{2,3}\d{3}` | Quarantine |
| `airline` | No | Non-empty and not the literal `'UNKNOWN'` | Impute as `'Unknown'` + quality flag (non-critical for duration/route KPIs) |
| `source` / `destination` | No | Non-empty, valid 3-letter city code, `source != destination` | Quarantine |
| `departure_time` / `arrival_time` | No | Parseable timestamp | Quarantine if unparseable |
| computed `duration_minutes` | — | `0 < duration <= 360` minutes (6 hrs — the longest domestic route observed here is ~5 hrs) after overnight adjustment | Flag `SUSPICIOUS` / quarantine if grossly negative even after adjustment |

### `bookings`

| Field | Nullable? | Rule | Failure action |
|---|---|---|---|
| `status` | No | One of `CONFIRMED`, `CANCELLED`, `PENDING` | Missing/`INVALID` → set to `'UNKNOWN'` + quality flag (booking itself still counted) |
| `flight_id`, `passenger_id` | No | Must exist in respective dimension | Quarantine (none observed) |

### `passengers`

| Field | Nullable? | Rule | Failure action |
|---|---|---|---|
| `last_name` | Yes (non-critical) | — | Keep, flag `quality_issue = MISSING_LAST_NAME` |
| `aadhaar_id`, `passport_number`, `phone`, `email` | No | PII — must never appear un-masked past the cleaning layer | Hash / mask (Section 6) |

### `payments`

| Field | Nullable? | Rule | Failure action |
|---|---|---|---|
| `amount` | No | Numeric, `> 0` | Missing or `'INVALID'` → quarantine the payment row (can't safely impute a transaction amount) |

**Duplicate definition (documented, per brief's ask):**
> *"A duplicate flight record is defined as two rows identical across every column (`flight_id`, `airline`, `source`, `destination`, `departure_time`, `arrival_time`). Rows that merely share a `flight_id` but differ in timestamp represent the same flight number operating on a different day, which is normal airline scheduling practice, and are kept."*


## 4. Validation Layer — Flights

Instead of `bad_row -> delete`, every row is tagged with `validation_status` and (if invalid) a `reason_code`, then either passed through to cleaning or routed to quarantine. Nothing is silently dropped.


In [7]:
def validate_flights(df: pd.DataFrame) -> pd.DataFrame:
    """Tags every flight row with validation_status / reason_code. Does not drop anything."""
    df = df.copy()
    df['row_ref'] = df.index  # traceability back to the raw file

    reasons = []
    for _, r in df.iterrows():
        row_reasons = []

        # Flight ID format check: 2-3 letter/digit airline code + 3 digit number
        fid = str(r['flight_id']).strip() if pd.notna(r['flight_id']) else ''
        if fid == '' :
            row_reasons.append('MISSING_FLIGHT_ID')
        elif not re.match(r'^[A-Z0-9]{2,3}\d{3}$', fid):
            row_reasons.append('INVALID_FLIGHT_ID_FORMAT')

        # Source / destination
        if pd.isna(r['source']) or str(r['source']).strip() == '':
            row_reasons.append('MISSING_SOURCE')
        if pd.isna(r['destination']) or str(r['destination']).strip() == '':
            row_reasons.append('MISSING_DESTINATION')
        if pd.notna(r['source']) and pd.notna(r['destination']) and r['source'] == r['destination']:
            row_reasons.append('SOURCE_EQUALS_DESTINATION')

        # Timestamps
        if pd.isna(r['departure_time']):
            row_reasons.append('MISSING_DEPARTURE_TIME')
        if pd.isna(r['arrival_time']):
            row_reasons.append('MISSING_ARRIVAL_TIME')

        # Gross timestamp corruption: gap more negative than a plausible overnight wrap
        # (a real overnight flight is arrival next-day within a few hours; anything where
        # the *raw* gap is worse than -12h signals a data-entry error, not a real flight)
        if pd.notna(r['departure_time']) and pd.notna(r['arrival_time']):
            raw_gap_minutes = (r['arrival_time'] - r['departure_time']).total_seconds() / 60
            if raw_gap_minutes < -12 * 60:
                row_reasons.append('CORRUPTED_TIMESTAMP')

        reasons.append(row_reasons)

    df['reason_codes'] = reasons
    df['is_valid'] = df['reason_codes'].apply(lambda x: len(x) == 0)
    # Exact full-row duplicates are also invalid (kept flagged, handled in cleaning)
    exact_dupe_mask = df.duplicated(subset=['flight_id','airline','source','destination','departure_time','arrival_time'], keep='first')
    df.loc[exact_dupe_mask, 'reason_codes'] = df.loc[exact_dupe_mask, 'reason_codes'].apply(lambda x: x + ['DUPLICATE_RECORD'])
    df.loc[exact_dupe_mask, 'is_valid'] = False

    df['validation_status'] = np.where(df['is_valid'], 'VALID', 'INVALID')
    return df

flights_validated = validate_flights(flights_raw)
print(flights_validated['validation_status'].value_counts())
print()
from collections import Counter
all_reasons = Counter(r for rs in flights_validated['reason_codes'] for r in rs)
print('Reason code breakdown:', dict(all_reasons))


validation_status
VALID      99
INVALID     1
Name: count, dtype: int64

Reason code breakdown: {'DUPLICATE_RECORD': 1}


Only `MISSING_FLIGHT_ID`/format/corruption/duplication are treated as *hard* validation failures that route a flight to quarantine. A missing/UNKNOWN airline is a *soft* issue — handled by imputation with a quality flag in Section 5, since it doesn't block duration or route KPIs.

In [8]:
# Split: hard failures -> quarantine candidates. Soft issue (airline) does not, by itself, invalidate the row.
HARD_REASONS = {'MISSING_FLIGHT_ID','INVALID_FLIGHT_ID_FORMAT','MISSING_SOURCE','MISSING_DESTINATION',
                'SOURCE_EQUALS_DESTINATION','MISSING_DEPARTURE_TIME','MISSING_ARRIVAL_TIME',
                'CORRUPTED_TIMESTAMP','DUPLICATE_RECORD'}

flights_validated['hard_reasons'] = flights_validated['reason_codes'].apply(lambda rs: [r for r in rs if r in HARD_REASONS])
flights_validated['route_to_quarantine'] = flights_validated['hard_reasons'].apply(lambda x: len(x) > 0)

flights_ok = flights_validated[~flights_validated['route_to_quarantine']].copy()
flights_quarantine = flights_validated[flights_validated['route_to_quarantine']].copy()

print(f'Passing hard validation : {len(flights_ok)}')
print(f'Routed to quarantine    : {len(flights_quarantine)}')


Passing hard validation : 99
Routed to quarantine    : 1


## 5. Cleaning & Standardization — Flights

Applied only to the rows that passed hard validation.


In [9]:
flights_clean = flights_ok.copy()

# --- Airline: standardize casing/whitespace, impute missing/UNKNOWN ---
flights_clean['airline'] = flights_clean['airline'].astype('object')
flights_clean['airline'] = flights_clean['airline'].apply(lambda x: str(x).strip() if pd.notna(x) else x)
flights_clean['airline_quality_flag'] = np.where(
    flights_clean['airline'].isna() | flights_clean['airline'].isin(['UNKNOWN', '']), 'IMPUTED_UNKNOWN_AIRLINE', 'OK'
)
flights_clean['airline'] = flights_clean['airline'].fillna('Unknown').replace({'UNKNOWN': 'Unknown', '': 'Unknown'})

# --- Source / destination: trim + uppercase (already clean, this is defensive standardization) ---
flights_clean['source'] = flights_clean['source'].astype(str).str.strip().str.upper()
flights_clean['destination'] = flights_clean['destination'].astype(str).str.strip().str.upper()
flights_clean['route'] = flights_clean['source'] + ' -> ' + flights_clean['destination']

print(flights_clean['airline'].value_counts())
print()
print(flights_clean['airline_quality_flag'].value_counts())


airline
Vistara      30
SpiceJet     24
IndiGo       20
Air India    16
Unknown       9
Name: count, dtype: int64

airline_quality_flag
OK                         90
IMPUTED_UNKNOWN_AIRLINE     9
Name: count, dtype: int64


## 6. Overnight Flight Handling & Duration Calculation

**Important finding from profiling:** unlike a typical "time-only" feed, `departure_time`/`arrival_time` in this source already carry a **full date + time**. That means a genuine overnight flight (e.g. departs `20-Apr 23:38`, arrives `21-Apr 02:32`) already has the *correct* next-day date recorded — there is nothing to "patch". The only rows where `arrival_time` is *chronologically before* `departure_time` by a large margin (we saw one case ~19 hours) are **corrupted records**, not real overnight flights — those were already routed to quarantine in Section 4 via `CORRUPTED_TIMESTAMP` (threshold: raw gap worse than -12h).

So the actual overnight-handling rule for *this* dataset is:

```
raw_gap = arrival_time - departure_time     # already date-aware
is_overnight = date(arrival_time) != date(departure_time)   # AND raw_gap > 0 (legitimate)
duration_minutes = raw_gap in minutes                        # no date patch needed here
```

We still implement the general "add a day if negative" defensive branch below — it is a no-op on this dataset (since dates are already correct) but keeps the pipeline correct if a future feed arrives as time-only fields without dates, which is the more common real-world version of this problem.


In [10]:
def compute_duration(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    raw_gap = (df['arrival_time'] - df['departure_time'])

    # Defensive branch: if a future feed ever supplies negative gaps because only
    # time-of-day was captured (no date), patch by rolling the arrival forward a day.
    needs_patch = raw_gap.dt.total_seconds() < 0
    arrival_adjusted = df['arrival_time'].copy()
    arrival_adjusted[needs_patch] = arrival_adjusted[needs_patch] + pd.Timedelta(days=1)

    df['arrival_time_adjusted'] = arrival_adjusted
    df['duration_minutes'] = (df['arrival_time_adjusted'] - df['departure_time']).dt.total_seconds() / 60
    # Business definition of "overnight": the flight's date of arrival differs from its date of departure.
    df['is_overnight'] = df['departure_time'].dt.date != df['arrival_time_adjusted'].dt.date
    return df

flights_clean = compute_duration(flights_clean)

print('Overnight flights (arrival date != departure date):', flights_clean['is_overnight'].sum())
print()
print(flights_clean['duration_minutes'].describe())


Overnight flights (arrival date != departure date): 29

count     99.000000
mean     140.929393
std       72.791952
min       30.000000
25%       87.500000
50%      126.000000
75%      197.500000
max      299.000000
Name: duration_minutes, dtype: float64


In [11]:
# Duration validation: classify VALID / SUSPICIOUS / INVALID.
# Threshold justification: the observed valid domestic flight durations here cluster
# between ~30 and ~300 minutes; the brief's routes (DEL/BOM/MAA/BLR/HYD/CCU) are all
# short/medium-haul domestic sectors, so we treat 0-360 min as VALID, negative or 0 as
# INVALID (should never occur post-overnight-fix -> would itself indicate a residual
# data issue), and > 360 min as SUSPICIOUS (kept, but flagged for review rather than dropped).
def classify_duration(mins):
    if mins <= 0:
        return 'INVALID'
    if mins > 360:
        return 'SUSPICIOUS'
    return 'VALID'

flights_clean['duration_status'] = flights_clean['duration_minutes'].apply(classify_duration)
print(flights_clean['duration_status'].value_counts())

# Any residual INVALID after overnight-adjustment gets routed to quarantine too.
residual_invalid = flights_clean[flights_clean['duration_status'] == 'INVALID'].copy()
if len(residual_invalid) > 0:
    residual_invalid['hard_reasons'] = [['INVALID_DURATION_POST_ADJUSTMENT']] * len(residual_invalid)
    flights_quarantine = pd.concat([flights_quarantine, residual_invalid], ignore_index=True)
    flights_clean = flights_clean[flights_clean['duration_status'] != 'INVALID'].copy()

print()
print('Final clean flights       :', len(flights_clean))
print('Final quarantined flights :', len(flights_quarantine))


duration_status
VALID    99
Name: count, dtype: int64

Final clean flights       : 99
Final quarantined flights : 1


### Delay / Anomaly strategy

The base fields provided (`flight_id`, `airline`, `source`, `destination`, `departure_time`, `arrival_time`) do **not** include a separate *scheduled* vs *actual* time — so there is no ground truth to compute a "delay in minutes" against. Fabricating one would misrepresent the data.

Instead, **anomaly** is used as defined by data-quality signals already computed:
- `CORRUPTED_TIMESTAMP` records (quarantined)
- `SUSPICIOUS` duration records (kept, flagged)
- `DUPLICATE_RECORD` (quarantined)
- `IMPUTED_UNKNOWN_AIRLINE` (kept, flagged)

An `anomaly_flag` is added to the curated table so Power BI can report an "anomaly rate" without inventing delay figures.


In [12]:
flights_clean['anomaly_flag'] = np.where(
    (flights_clean['duration_status'] == 'SUSPICIOUS') | (flights_clean['airline_quality_flag'] != 'OK'),
    'ANOMALY', 'NORMAL'
)
print(flights_clean['anomaly_flag'].value_counts())


anomaly_flag
NORMAL     90
ANOMALY     9
Name: count, dtype: int64


## 7. PII Masking — Passengers & Bookings

The `passengers` table carries genuine PII (`aadhaar_id`, `phone`, `email`) and `bookings` carries `passport_number` and an emergency contact name/phone. Per the brief's requirement, none of this may reach the analytical/reporting layer un-masked.

**Approach:** irreversible SHA-256 hashing (salted) for identifiers that only need to support joins/counts (`aadhaar_id`, `phone`, `passport_number`), and partial masking for fields that are still useful to *display* in a support/audit context (`email`).


In [13]:
SALT = 'asg-airlines-2026'  # in production this would live in a secrets manager, never in code

def hash_value(value, salt=SALT):
    if pd.isna(value):
        return None
    return hashlib.sha256(f'{salt}{value}'.encode()).hexdigest()[:16]

def mask_email(email):
    if pd.isna(email):
        return None
    try:
        local, domain = str(email).split('@')
        masked_local = local[0] + '*' * max(len(local) - 2, 1) + (local[-1] if len(local) > 1 else '')
        return f'{masked_local}@{domain}'
    except ValueError:
        return 'INVALID_EMAIL'

def mask_phone(phone):
    if pd.isna(phone):
        return None
    s = str(phone)
    return s[:4] + '*' * (len(s) - 6) + s[-2:] if len(s) > 6 else '*' * len(s)

passengers_clean = passengers_raw.copy()
passengers_clean['quality_flag'] = np.where(passengers_clean['last_name'].isna(), 'MISSING_LAST_NAME', 'OK')
passengers_clean['last_name'] = passengers_clean['last_name'].fillna('Unknown')

passengers_clean['aadhaar_id_masked'] = passengers_clean['aadhaar_id'].apply(hash_value)
passengers_clean['phone_masked']      = passengers_clean['phone'].apply(mask_phone)
passengers_clean['email_masked']      = passengers_clean['email'].apply(mask_email)

# Drop raw PII columns entirely from the analytics-facing table
passengers_curated = passengers_clean.drop(columns=['aadhaar_id', 'phone', 'email']).rename(
    columns={'aadhaar_id_masked': 'aadhaar_id_hash', 'phone_masked': 'phone_masked', 'email_masked': 'email_masked'}
)

print('Sample of masked passenger data:')
passengers_curated[['passenger_id','first_name','last_name','aadhaar_id_hash','phone_masked','email_masked']].head()


Sample of masked passenger data:


,passenger_id,first_name,last_name,aadhaar_id_hash,phone_masked,email_masked
0,SP0001,Passenger001,Sample001,6289a88460cc2b39,+91-********01,p**********1@example.com
1,SP0002,Passenger002,Sample002,2ec46d30809f0310,+91-********02,p**********2@example.com
2,SP0003,Passenger003,Sample003,e6ca554d73c90f4e,+91-********03,p**********3@example.com
3,SP0004,Passenger004,Sample004,7813156e61ebf32b,+91-********04,p**********4@example.com
4,SP0005,Passenger005,Sample005,6cfc3e77a28783fb,+91-********05,p**********5@example.com


In [14]:
bookings_clean = bookings_raw.copy()

# status cleanup
bookings_clean['status_quality_flag'] = np.where(
    bookings_clean['status'].isna() | (bookings_clean['status'] == 'INVALID'), 'IMPUTED_UNKNOWN_STATUS', 'OK'
)
bookings_clean['status'] = bookings_clean['status'].fillna('UNKNOWN').replace({'INVALID': 'UNKNOWN'})

# PII masking on bookings
bookings_clean['passport_number_hash'] = bookings_clean['passport_number'].apply(hash_value)
bookings_clean['emergency_contact_phone_masked'] = bookings_clean['emergency_contact_phone'].apply(mask_phone)
# Emergency contact name is direct PII with no analytical value at all -> mask fully, don't even hash-preserve it
bookings_clean['emergency_contact_name_masked'] = bookings_clean['emergency_contact_name'].apply(
    lambda x: (str(x).split()[0][0] + '.' if pd.notna(x) and len(str(x).split()) > 0 else None)
)

bookings_curated = bookings_clean.drop(columns=['passport_number', 'emergency_contact_name', 'emergency_contact_phone'])

print(bookings_clean['status'].value_counts())
bookings_curated.head(3)


status
CANCELLED    36
CONFIRMED    32
PENDING      28
UNKNOWN       4
Name: count, dtype: int64


,booking_id,passenger_id,flight_id,booking_date,status,seat_number,status_quality_flag,passport_number_hash,emergency_contact_phone_masked,emergency_contact_name_masked
0,B1000,SP0001,AI192,2025-06-14 11:37:36.951,CANCELLED,3D,OK,721ffddc6fc3cfc1,+91-********01,C.
1,B1001,SP0001,6F026,2025-11-02 11:37:36.951,CANCELLED,18A,OK,b0acd936bf201d68,+91-********02,C.
2,B1002,SP0085,SJ010,2025-08-25 11:37:36.951,CANCELLED,30C,OK,8d8bfaa0f077d07a,+91-********03,C.


**Privacy & Access Control note:** the dataset *does* contain passenger-level PII (Aadhaar ID, passport number, phone, email, emergency contact). The above hashing/masking is applied before any of this data reaches the curated/analytics layer that Power BI would connect to. In a production Azure setup, the recommended controls would additionally include: Azure Key Vault for the hashing salt/keys, row-level security in the Power BI semantic model so only authorized operations roles see passenger-level tables at all (the flight-ops dashboards only need the fully-anonymous `FactFlights`/`FactBookings` — not passenger identity), and column-level encryption at rest for the raw layer.


## 8. Cleaning — Payments

In [15]:
payments_clean = payments_raw.copy()
payments_clean['amount_numeric'] = pd.to_numeric(payments_clean['amount'], errors='coerce')

payments_clean['validation_status'] = np.where(
    payments_clean['amount_numeric'].isna() | (payments_clean['amount_numeric'] <= 0),
    'INVALID', 'VALID'
)
payments_clean['reason_code'] = np.where(
    payments_clean['amount'].isna(), 'MISSING_AMOUNT',
    np.where(payments_clean['amount'].astype(str) == 'INVALID', 'NON_NUMERIC_AMOUNT', None)
)

payments_quarantine = payments_clean[payments_clean['validation_status'] == 'INVALID'].copy()
payments_curated = payments_clean[payments_clean['validation_status'] == 'VALID'].drop(
    columns=['amount', 'reason_code']
).rename(columns={'amount_numeric': 'amount'})

print('Valid payments      :', len(payments_curated))
print('Quarantined payments:', len(payments_quarantine))
print()
print(payments_quarantine['reason_code'].value_counts())


Valid payments      : 86
Quarantined payments: 8

reason_code
NON_NUMERIC_AMOUNT    4
MISSING_AMOUNT        4
Name: count, dtype: int64


## 9. Dimensional Model

```
                    DimAirline
                        |
DimLocation ------ FactFlights ------ DimDate
                        |
                  (passenger side)
                        |
                  FactBookings ------ DimPassenger
                        |
                  FactPayments
```

- **FactFlights** — one row per flight event: measures are `duration_minutes`, `is_overnight`, `anomaly_flag`.
- **DimAirline** — airline reference.
- **DimRoute** (source → destination pairs).
- **DimDate** — standard calendar dimension derived from `departure_time`.
- **FactBookings** — one row per booking, links to `FactFlights` and `DimPassenger`.
- **FactPayments** — one row per payment, links to `FactBookings`.
- **DimPassenger** — masked passenger attributes only (no raw PII).


In [16]:
# --- flight_key: surrogate key for the fact table ---
flights_clean = flights_clean.reset_index(drop=True)
flights_clean['flight_key'] = 'FK' + (flights_clean.index + 1).astype(str).str.zfill(5)

# --- DimAirline ---
dim_airline = pd.DataFrame({'airline': sorted(flights_clean['airline'].unique())})
dim_airline['airline_key'] = 'AL' + (dim_airline.index + 1).astype(str).str.zfill(3)
dim_airline = dim_airline[['airline_key', 'airline']]

# --- DimRoute ---
dim_route = flights_clean[['source', 'destination', 'route']].drop_duplicates().reset_index(drop=True)
dim_route['route_key'] = 'RT' + (dim_route.index + 1).astype(str).str.zfill(3)
dim_route = dim_route[['route_key', 'source', 'destination', 'route']]

# --- DimDate (from departure date) ---
all_dates = pd.date_range(flights_clean['departure_time'].dt.date.min(), flights_clean['departure_time'].dt.date.max())
dim_date = pd.DataFrame({'date': all_dates})
dim_date['date_key'] = dim_date['date'].dt.strftime('%Y%m%d')
dim_date['year'] = dim_date['date'].dt.year
dim_date['month'] = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['day'] = dim_date['date'].dt.day
dim_date['day_name'] = dim_date['date'].dt.strftime('%A')
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek >= 5

# --- FactFlights ---
fact_flights = flights_clean.merge(dim_airline, on='airline', how='left') \
                             .merge(dim_route, on=['source','destination','route'], how='left')
fact_flights['date_key'] = fact_flights['departure_time'].dt.strftime('%Y%m%d')

fact_flights = fact_flights[[
    'flight_key','flight_id','airline_key','route_key','date_key',
    'departure_time','arrival_time','arrival_time_adjusted',
    'duration_minutes','is_overnight','duration_status','anomaly_flag','airline_quality_flag'
]]

print('DimAirline:', dim_airline.shape)
print('DimRoute  :', dim_route.shape)
print('DimDate   :', dim_date.shape)
print('FactFlights:', fact_flights.shape)
fact_flights.head(3)


DimAirline: (5, 2)
DimRoute  : (27, 4)
DimDate   : (1, 8)
FactFlights: (99, 13)


,flight_key,flight_id,airline_key,route_key,date_key,departure_time,arrival_time,arrival_time_adjusted,duration_minutes,is_overnight,duration_status,anomaly_flag,airline_quality_flag
0,FK00001,SJ010,AL003,RT001,20260420,2026-04-20 23:38:41.701,2026-04-21 02:32:41.701,2026-04-21 02:32:41.701,174.0,True,VALID,NORMAL,OK
1,FK00002,AI155,AL001,RT002,20260420,2026-04-20 23:35:41.703,2026-04-21 01:23:41.703,2026-04-21 01:23:41.703,108.0,True,VALID,NORMAL,OK
2,FK00003,UK094,AL005,RT002,20260420,2026-04-20 23:26:41.702,2026-04-21 01:11:41.702,2026-04-21 01:11:41.702,105.0,True,VALID,NORMAL,OK


In [17]:
# --- DimPassenger (masked) ---
dim_passenger = passengers_curated.copy()

# --- FactBookings ---
fact_bookings = bookings_curated.merge(
    flights_clean[['flight_id','flight_key']], on='flight_id', how='left'
)
fact_bookings = fact_bookings[[
    'booking_id','passenger_id','flight_key','flight_id','booking_date','status',
    'status_quality_flag','seat_number','passport_number_hash',
    'emergency_contact_name_masked','emergency_contact_phone_masked'
]]

# --- FactPayments ---
fact_payments = payments_curated.copy()

print('DimPassenger :', dim_passenger.shape)
print('FactBookings :', fact_bookings.shape)
print('FactPayments :', fact_payments.shape)


DimPassenger : (200, 10)
FactBookings : (100, 11)
FactPayments : (86, 5)


## 10. Quarantine Dataset

In [18]:
def build_quarantine_output(df, reasons_col='hard_reasons'):
    q = df.copy()
    q['reason_description'] = q[reasons_col].apply(lambda x: '; '.join(x) if isinstance(x, list) else str(x))
    keep_cols = [c for c in ['row_ref','flight_id','airline','source','destination',
                              'departure_time','arrival_time','reason_description'] if c in q.columns]
    return q[keep_cols]

flights_quarantine_out = build_quarantine_output(flights_quarantine)
flights_quarantine_out['validation_status'] = 'QUARANTINED'
print('Flights quarantined:', len(flights_quarantine_out))
flights_quarantine_out


Flights quarantined: 1


,row_ref,flight_id,airline,source,destination,departure_time,arrival_time,reason_description,validation_status
84,84,AI242,UNKNOWN,BLR,CCU,2026-04-20 16:41:41.704,2026-04-20 17:43:41.704,DUPLICATE_RECORD,QUARANTINED


In [19]:
payments_quarantine_out = payments_quarantine[['payment_id','booking_id','amount','payment_method','reason_code']].copy()
payments_quarantine_out['validation_status'] = 'QUARANTINED'
print('Payments quarantined:', len(payments_quarantine_out))
payments_quarantine_out.head()


Payments quarantined: 8


,payment_id,booking_id,amount,payment_method,reason_code,validation_status
1,PAY1033,B1012,INVALID,CARD,NON_NUMERIC_AMOUNT,QUARANTINED
14,PAY1184,B1019,NaN,UPI,MISSING_AMOUNT,QUARANTINED
29,PAY1283,B1118,NaN,NETBANKING,MISSING_AMOUNT,QUARANTINED
48,PAY1534,B1010,INVALID,NETBANKING,NON_NUMERIC_AMOUNT,QUARANTINED
64,PAY1664,B1033,NaN,UPI,MISSING_AMOUNT,QUARANTINED


## 11. Business KPIs

Computed directly from the curated `FactFlights` (and bookings/payments where relevant) — exactly the metrics the brief requires, plus a Data Quality Score as a differentiator.


In [20]:
# 1. Total Flights
total_flights = len(fact_flights)

# 2. Average Flight Duration
avg_duration = fact_flights.loc[fact_flights['duration_status']=='VALID', 'duration_minutes'].mean()

# 3. Route-wise traffic
route_traffic = fact_flights.merge(dim_route, on='route_key').groupby('route').size().sort_values(ascending=False)

# 4. Airline distribution
airline_dist = fact_flights.merge(dim_airline, on='airline_key').groupby('airline').size().sort_values(ascending=False)

# 5. Anomaly rate
anomaly_rate = (fact_flights['anomaly_flag'] == 'ANOMALY').mean() * 100

print(f'Total Flights            : {total_flights}')
print(f'Average Flight Duration  : {avg_duration:.1f} minutes')
print(f'Anomaly Rate             : {anomaly_rate:.2f}%')
print()
print('Top 10 routes by traffic:')
print(route_traffic.head(10))
print()
print('Airline distribution:')
print(airline_dist)


Total Flights            : 99
Average Flight Duration  : 140.9 minutes
Anomaly Rate             : 9.09%

Top 10 routes by traffic:
route
BOM -> CCU    12
BLR -> BOM     9
BOM -> DEL     7
HYD -> MAA     6
CCU -> DEL     5
BOM -> HYD     5
CCU -> HYD     5
DEL -> HYD     5
MAA -> BLR     5
MAA -> HYD     4
dtype: int64

Airline distribution:
airline
Vistara      30
SpiceJet     24
IndiGo       20
Air India    16
Unknown       9
dtype: int64


In [21]:
# Additional KPIs beyond the minimum ask
overnight_pct = fact_flights['is_overnight'].mean() * 100
avg_duration_by_route = fact_flights.merge(dim_route, on='route_key').groupby('route')['duration_minutes'].mean().sort_values(ascending=False)

# Revenue KPIs (from payments + bookings, confirmed bookings only)
confirmed_bookings = fact_bookings[fact_bookings['status'] == 'CONFIRMED']
revenue_df = confirmed_bookings.merge(fact_payments, on='booking_id', how='inner')
total_revenue = revenue_df['amount'].sum()
revenue_by_method = revenue_df.groupby('payment_method')['amount'].sum().sort_values(ascending=False)

print(f'Overnight flight share        : {overnight_pct:.1f}%')
print(f'Total confirmed-booking revenue: Rs. {total_revenue:,.2f}')
print()
print('Revenue by payment method:')
print(revenue_by_method)
print()
print('Average duration by route (mins):')
print(avg_duration_by_route)


Overnight flight share        : 29.3%
Total confirmed-booking revenue: Rs. 236,690.48

Revenue by payment method:
payment_method
UPI           160145.42
CARD           46837.98
NETBANKING     29707.08
Name: amount, dtype: float64

Average duration by route (mins):
route
MAA -> BOM    269.000000
CCU -> BLR    264.000000
HYD -> BLR    248.000000
DEL -> MAA    227.500000
HYD -> DEL    193.666667
CCU -> MAA    172.000000
CCU -> HYD    171.800000
MAA -> BLR    167.600000
CCU -> DEL    166.800000
BOM -> DEL    166.142857
BOM -> HYD    154.200000
DEL -> BOM    149.000000
MAA -> HYD    138.500000
BOM -> CCU    138.000000
HYD -> BOM    137.666667
BOM -> MAA    137.666667
DEL -> HYD    121.600000
BOM -> BLR    109.000000
MAA -> DEL    108.666667
BLR -> BOM    108.333885
BLR -> MAA    101.000000
HYD -> MAA    100.500828
BLR -> CCU     95.000000
HYD -> CCU     91.000000
MAA -> CCU     58.500000
DEL -> CCU     56.500000
DEL -> BLR     41.000000
Name: duration_minutes, dtype: float64


### ⭐ Data Quality Score

A single composite metric operations teams can use to gauge the reliability of the dataset *before* consuming the analytics — the differentiator recommended for this build.

```
DQ Score = 0.30 * Completeness + 0.30 * Validity + 0.20 * Uniqueness + 0.20 * Consistency
```

- **Completeness** — % of required fields populated across all raw flight rows.
- **Validity** — % of raw flight rows that passed hard validation (Section 4).
- **Uniqueness** — % of raw flight rows that are not exact duplicates.
- **Consistency** — % of validated rows whose computed duration is `VALID` (not `SUSPICIOUS`/`INVALID`) after overnight adjustment.

Weights are documented, not tuned — validity and completeness are weighted highest because a downstream KPI (duration, route traffic) is only as good as whether the record could be parsed and matched in the first place.


In [22]:
required_cols = ['flight_id','airline','source','destination','departure_time','arrival_time']
completeness = flights_raw[required_cols].notna().mean().mean() * 100
# airline UNKNOWN literal counts as incomplete too
completeness_adj = ((flights_raw[required_cols].notna()) & (flights_raw[required_cols] != 'UNKNOWN')).mean().mean() * 100

validity = (len(flights_raw) - len(flights_quarantine)) / len(flights_raw) * 100
uniqueness = (1 - flights_raw.duplicated(subset=['flight_id','airline','source','destination','departure_time','arrival_time']).mean()) * 100
consistency = (fact_flights['duration_status'] == 'VALID').mean() * 100

dq_score = 0.30*completeness_adj + 0.30*validity + 0.20*uniqueness + 0.20*consistency

print(f'Completeness : {completeness_adj:.1f}%')
print(f'Validity     : {validity:.1f}%')
print(f'Uniqueness   : {uniqueness:.1f}%')
print(f'Consistency  : {consistency:.1f}%')
print(f'--------------------------------')
print(f'DATA QUALITY SCORE : {dq_score:.1f}%')


Completeness : 98.3%
Validity     : 99.0%
Uniqueness   : 99.0%
Consistency  : 100.0%
--------------------------------
DATA QUALITY SCORE : 99.0%


## 12. Write Cleaned / Curated / Quarantine Outputs

In [23]:
# --- cleaned layer (post-validation, pre-modelling, still 1 table per source) ---
flights_clean.to_csv(CLEANED_DIR / 'flights_cleaned.csv', index=False)
bookings_curated.to_csv(CLEANED_DIR / 'bookings_cleaned.csv', index=False)
passengers_curated.to_csv(CLEANED_DIR / 'passengers_cleaned_masked.csv', index=False)
payments_curated.to_csv(CLEANED_DIR / 'payments_cleaned.csv', index=False)

# --- curated / star-schema layer (Power BI-ready) ---
fact_flights.to_csv(CURATED_DIR / 'fact_flights.csv', index=False)
dim_airline.to_csv(CURATED_DIR / 'dim_airline.csv', index=False)
dim_route.to_csv(CURATED_DIR / 'dim_route.csv', index=False)
dim_date.to_csv(CURATED_DIR / 'dim_date.csv', index=False)
dim_passenger.to_csv(CURATED_DIR / 'dim_passenger.csv', index=False)
fact_bookings.to_csv(CURATED_DIR / 'fact_bookings.csv', index=False)
fact_payments.to_csv(CURATED_DIR / 'fact_payments.csv', index=False)

# KPI summary tables (so Power BI, or anyone, can sanity-check against the notebook)
kpi_summary = pd.DataFrame([
    {'kpi': 'Input Flights', 'value': len(flights_raw)},
    {'kpi': 'Total Flights', 'value': total_flights},
    {'kpi': 'Quarantined Flights', 'value': len(flights_quarantine_out)},
    {'kpi': 'Average Flight Duration (min)', 'value': round(avg_duration, 2)},
    {'kpi': 'Anomaly Rate (%)', 'value': round(anomaly_rate, 2)},
    {'kpi': 'Overnight Flight Share (%)', 'value': round(overnight_pct, 2)},
    {'kpi': 'Total Confirmed Revenue (Rs.)', 'value': round(total_revenue, 2)},
    {'kpi': 'Data Quality Score (%)', 'value': round(dq_score, 2)},
])
kpi_summary.to_csv(CURATED_DIR / 'kpi_summary.csv', index=False)

dq_metrics = pd.DataFrame([{
    'completeness': completeness, 'validity': validity, 'uniqueness': uniqueness,
    'consistency': consistency, 'dq_score': dq_score
}])
dq_metrics.to_csv(CURATED_DIR / 'dq_metrics.csv', index=False)
record_quality_summary = pd.DataFrame([
    {'status': 'VALID', 'record_count': len(fact_flights)},
    {'status': 'INVALID', 'record_count': len(flights_quarantine_out)}
])
record_quality_summary.to_csv(CURATED_DIR / 'record_quality_summary.csv', index=False)
route_traffic.reset_index().rename(columns={0:'flight_count'}).to_csv(CURATED_DIR / 'kpi_route_traffic.csv', index=False)
airline_dist.reset_index().rename(columns={0:'flight_count'}).to_csv(CURATED_DIR / 'kpi_airline_distribution.csv', index=False)

# --- quarantine layer ---
flights_quarantine_out.to_csv(QUARANTINE_DIR / 'flights_quarantine.csv', index=False)
payments_quarantine_out.to_csv(QUARANTINE_DIR / 'payments_quarantine.csv', index=False)

print('All outputs written.')
print()
print('cleaned/   :', [p.name for p in CLEANED_DIR.glob('*.csv')])
print('curated/   :', [p.name for p in CURATED_DIR.glob('*.csv')])
print('quarantine/:', [p.name for p in QUARANTINE_DIR.glob('*.csv')])


All outputs written.

cleaned/   : ['bookings_cleaned.csv', 'flights_cleaned.csv', 'passengers_cleaned_masked.csv', 'payments_cleaned.csv']
curated/   : ['dim_airline.csv', 'dim_date.csv', 'kpi_route_traffic.csv', 'dim_passenger.csv', 'kpi_airline_distribution.csv', 'fact_payments.csv', 'fact_flights.csv', 'dim_route.csv', 'kpi_summary.csv', 'fact_bookings.csv']
quarantine/: ['flights_quarantine.csv', 'payments_quarantine.csv']


## 13. Testing

Unit tests for the core transformation logic — overnight handling, missing values, malformed IDs, duplicates — run against small, hand-built fixtures (not the production data), so each rule can be verified in isolation.


In [24]:
test_results = []

def run_test(name, condition, expected, actual):
    status = 'PASS' if condition else 'FAIL'
    test_results.append({'test': name, 'expected': expected, 'actual': actual, 'status': status})

# Test 1: Normal same-day flight
sample = pd.DataFrame({
    'flight_id': ['AI999'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.Timestamp('2026-01-01 10:00:00')],
    'arrival_time':   [pd.Timestamp('2026-01-01 12:00:00')],
})
res = compute_duration(sample)
run_test('T1: same-day flight duration', res['duration_minutes'].iloc[0] == 120 and not res['is_overnight'].iloc[0],
          '120 min, not overnight', f"{res['duration_minutes'].iloc[0]} min, overnight={res['is_overnight'].iloc[0]}")

# Test 2a: Overnight flight where the source data ALREADY carries the correct next-day date
# (this is how the real ASG dataset is structured)
sample2 = pd.DataFrame({
    'flight_id': ['AI998'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.Timestamp('2026-01-01 23:40:00')],
    'arrival_time':   [pd.Timestamp('2026-01-02 01:20:00')],
})
res2 = compute_duration(sample2)
run_test('T2a: overnight (date already correct) duration', res2['duration_minutes'].iloc[0] == 100 and res2['is_overnight'].iloc[0],
          '100 min, overnight=True', f"{res2['duration_minutes'].iloc[0]} min, overnight={res2['is_overnight'].iloc[0]}")

# Test 2b: Defensive branch - a time-only feed where arrival is stored as "earlier" because
# no date rollover was applied upstream; the pipeline must patch it forward a day.
sample2b = pd.DataFrame({
    'flight_id': ['AI988'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.Timestamp('2026-01-01 23:40:00')],
    'arrival_time':   [pd.Timestamp('2026-01-01 01:20:00')],  # same date, time-only rollover missing
})
res2b = compute_duration(sample2b)
run_test('T2b: overnight (defensive date-patch) duration', res2b['duration_minutes'].iloc[0] == 100 and res2b['is_overnight'].iloc[0],
          '100 min, overnight=True', f"{res2b['duration_minutes'].iloc[0]} min, overnight={res2b['is_overnight'].iloc[0]}")

# Test 3: Missing departure time -> validation flags it
sample3 = pd.DataFrame({
    'flight_id': ['AI997'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.NaT], 'arrival_time': [pd.Timestamp('2026-01-01 12:00:00')],
})
res3 = validate_flights(sample3)
run_test('T3: missing departure_time flagged', 'MISSING_DEPARTURE_TIME' in res3['reason_codes'].iloc[0],
          'MISSING_DEPARTURE_TIME in reasons', res3['reason_codes'].iloc[0])

# Test 4: Missing arrival time
sample4 = pd.DataFrame({
    'flight_id': ['AI996'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.Timestamp('2026-01-01 10:00:00')], 'arrival_time': [pd.NaT],
})
res4 = validate_flights(sample4)
run_test('T4: missing arrival_time flagged', 'MISSING_ARRIVAL_TIME' in res4['reason_codes'].iloc[0],
          'MISSING_ARRIVAL_TIME in reasons', res4['reason_codes'].iloc[0])

# Test 5: Malformed flight ID
sample5 = pd.DataFrame({
    'flight_id': ['??-bad-id'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.Timestamp('2026-01-01 10:00:00')], 'arrival_time': [pd.Timestamp('2026-01-01 12:00:00')],
})
res5 = validate_flights(sample5)
run_test('T5: malformed flight_id flagged', 'INVALID_FLIGHT_ID_FORMAT' in res5['reason_codes'].iloc[0],
          'INVALID_FLIGHT_ID_FORMAT in reasons', res5['reason_codes'].iloc[0])

# Test 6: Duplicate record
sample6 = pd.concat([sample, sample], ignore_index=True)
res6 = validate_flights(sample6)
run_test('T6: exact duplicate flagged on 2nd row', 'DUPLICATE_RECORD' in res6['reason_codes'].iloc[1],
          'DUPLICATE_RECORD on row 2', res6['reason_codes'].iloc[1])

# Test 7: Corrupted timestamp (arrival far before departure, not a real overnight)
sample7 = pd.DataFrame({
    'flight_id': ['AI995'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['BOM'],
    'departure_time': [pd.Timestamp('2026-01-02 18:00:00')], 'arrival_time': [pd.Timestamp('2026-01-01 20:00:00')],
})
res7 = validate_flights(sample7)
run_test('T7: corrupted timestamp flagged', 'CORRUPTED_TIMESTAMP' in res7['reason_codes'].iloc[0],
          'CORRUPTED_TIMESTAMP in reasons', res7['reason_codes'].iloc[0])

# Test 8: Suspicious (very long) duration classification
run_test('T8: suspicious duration classification', classify_duration(400) == 'SUSPICIOUS',
          'SUSPICIOUS', classify_duration(400))

# Test 9: source == destination
sample9 = pd.DataFrame({
    'flight_id': ['AI994'], 'airline': ['Air India'], 'source': ['DEL'], 'destination': ['DEL'],
    'departure_time': [pd.Timestamp('2026-01-01 10:00:00')], 'arrival_time': [pd.Timestamp('2026-01-01 12:00:00')],
})
res9 = validate_flights(sample9)
run_test('T9: source==destination flagged', 'SOURCE_EQUALS_DESTINATION' in res9['reason_codes'].iloc[0],
          'SOURCE_EQUALS_DESTINATION in reasons', res9['reason_codes'].iloc[0])

# Test 10: Pipeline rerun determinism (re-running validate+clean on the same data gives identical counts)
rerun = compute_duration(validate_flights(flights_raw))
run_test('T10: pipeline rerun determinism', rerun['is_overnight'].sum() == flights_validated.pipe(
            lambda d: compute_duration(d)['is_overnight'].sum()),
          'identical overnight count on rerun', rerun['is_overnight'].sum())

test_df = pd.DataFrame(test_results)
print(test_df.to_string(index=False))
print()
print(f"{(test_df['status']=='PASS').sum()} / {len(test_df)} tests passed")


                                          test                             expected                      actual status
                  T1: same-day flight duration               120 min, not overnight  120.0 min, overnight=False   PASS
T2a: overnight (date already correct) duration              100 min, overnight=True   100.0 min, overnight=True   PASS
T2b: overnight (defensive date-patch) duration              100 min, overnight=True   100.0 min, overnight=True   PASS
            T3: missing departure_time flagged    MISSING_DEPARTURE_TIME in reasons    [MISSING_DEPARTURE_TIME]   PASS
              T4: missing arrival_time flagged      MISSING_ARRIVAL_TIME in reasons      [MISSING_ARRIVAL_TIME]   PASS
               T5: malformed flight_id flagged  INVALID_FLIGHT_ID_FORMAT in reasons  [INVALID_FLIGHT_ID_FORMAT]   PASS
        T6: exact duplicate flagged on 2nd row            DUPLICATE_RECORD on row 2          [DUPLICATE_RECORD]   PASS
               T7: corrupted timestamp flagged  

## 14. Pipeline Run Log

In [25]:
print('=' * 50)
print('ASG AIRLINES PIPELINE - RUN SUMMARY')
print('=' * 50)
print()
print(f'Input records (flights)      : {len(flights_raw)}')
print()
print('Validation')
print('-' * 30)
print(f'Missing/UNKNOWN airline      : {(flights_raw["airline"].isna() | (flights_raw["airline"]=="UNKNOWN")).sum()}')
print(f'Duplicate records            : {flights_raw.duplicated().sum()}')
print(f'Corrupted timestamps         : {sum(1 for rs in flights_validated["reason_codes"] if "CORRUPTED_TIMESTAMP" in rs)}')
print(f'Total quarantined (flights)  : {len(flights_quarantine_out)}')
print()
print('Transformation')
print('-' * 30)
print(f'Overnight flights corrected  : {fact_flights["is_overnight"].sum()}')
print(f'Valid-duration flights       : {(fact_flights["duration_status"]=="VALID").sum()}')
print(f'Anomalies flagged            : {(fact_flights["anomaly_flag"]=="ANOMALY").sum()}')
print()
print('Output')
print('-' * 30)
print(f'Curated flight records       : {len(fact_flights)}')
print(f'Quarantined flight records   : {len(flights_quarantine_out)}')
print(f'Quarantined payment records  : {len(payments_quarantine_out)}')
print(f'Data Quality Score           : {dq_score:.1f}%')
print()
print(f'Tests passed                 : {(test_df["status"]=="PASS").sum()} / {len(test_df)}')
print()
print('Pipeline status              : SUCCESS')
print('=' * 50)


ASG AIRLINES PIPELINE - RUN SUMMARY

Input records (flights)      : 100

Validation
------------------------------
Missing/UNKNOWN airline      : 10
Duplicate records            : 1
Corrupted timestamps         : 0
Total quarantined (flights)  : 1

Transformation
------------------------------
Overnight flights corrected  : 29
Valid-duration flights       : 99
Anomalies flagged            : 9

Output
------------------------------
Curated flight records       : 99
Quarantined flight records   : 1
Quarantined payment records  : 8
Data Quality Score           : 99.0%

Tests passed                 : 11 / 11

Pipeline status              : SUCCESS
